In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Markdown / Wiki Retrieval Assistant

## 1. Project Overview

**Task:** Ingest a collection of markdown/wiki-style documents, parse their structure (headings, sections, metadata), build a clean vector index with rich metadata, and answer questions over them with cited section-level sources.

**Focus areas:**
- Markdown-aware splitting (respect heading hierarchy, not arbitrary character cuts)
- Document metadata extraction (title, section path, heading level, word count)
- Metadata-filtered retrieval (by document, section depth, topic)
- Section-level citations in answers

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store with metadata filtering

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Markdown-aware chunking** — split on heading boundaries instead of arbitrary character counts |
| 2 | **Section path tracking** — maintain the full heading hierarchy (`Doc > H1 > H2 > H3`) |
| 3 | **Rich metadata indexing** — store document name, section path, heading level, word count |
| 4 | **Metadata-filtered retrieval** — query within a specific document or heading level |
| 5 | **Section-cited QA** — answers that reference `[Doc: title > section]` |
| 6 | **Retrieval evaluation** — Recall@k and MRR on ground-truth question-section pairs |

## 3. Problem Statement

Teams accumulate knowledge in wikis, READMEs, runbooks, and markdown documentation. Finding the right section across dozens of documents is painful:

- **Keyword search** misses paraphrased concepts
- **Full-document retrieval** returns too much context — you need the *section*, not the whole page
- **Naive chunking** (split every 500 chars) breaks mid-sentence and loses heading context

This notebook builds a retrieval assistant that understands markdown structure, indexes at the section level, and returns precise answers with section-level citations.

## 4. Why Markdown-Aware Indexing

| Approach | Problem |
|----------|--------|
| Character-based splitting | Cuts mid-paragraph; chunks have no structural meaning |
| Sentence splitting | Loses heading context; can't filter by section |
| Page-level indexing | Too coarse; retrieves irrelevant sections along with relevant ones |
| **Heading-based splitting** | **Preserves document structure; each chunk maps to a meaningful section** |

## 5. Pipeline Architecture

```
Markdown Documents (wiki pages, READMEs, runbooks)
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 1: PARSE MARKDOWN STRUCTURE                 │
  │  • Split on heading boundaries (H1, H2, H3)        │
  │  • Track heading hierarchy as section path          │
  │  • Extract metadata: doc title, level, word count   │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 2: EMBED + INDEX                            │
  │  • Embed section text with MiniLM                  │
  │  • Store in ChromaDB with metadata fields           │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 3: RETRIEVE + ANSWER                        │
  │  • Semantic query + optional metadata filters       │
  │  • Build context from top-k sections                │
  │  • Generate answer with [Doc: title > section] cites│
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 4: EVALUATE                                 │
  │  • Recall@k and MRR on ground-truth pairs           │
  │  • Quality scoring via LLM-as-judge                 │
  └────────────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Retrieval top-k: {TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Synthetic Wiki Corpus

## 10. Build the Document Collection

We create 5 wiki-style markdown documents covering a fictional internal engineering wiki. Each document has nested headings (H1 → H2 → H3), covering topics a team would actually look up.

In [ ]:
WIKI_DOCS = {
    "Deployment Guide": """# Deployment Guide

This guide covers how to deploy services to our Kubernetes cluster.

## Prerequisites

Before deploying, ensure you have:
- `kubectl` installed and configured for the target cluster
- Docker images pushed to our container registry (`registry.internal.io`)
- A valid `kubeconfig` file — ask the platform team on #infra-access if you need one
- Helm 3.x installed (we do not support Helm 2 tiller-based deployments)

## Standard Deployment Process

### Step 1 — Build and Push Image

Build the Docker image using the project Dockerfile and push to our registry:

```bash
docker build -t registry.internal.io/myapp:v1.2.3 .
docker push registry.internal.io/myapp:v1.2.3
```

Always tag with the exact version. Never use `latest` in production.

### Step 2 — Update Helm Values

Edit the `values.yaml` in your service's Helm chart directory:

```yaml
image:
  repository: registry.internal.io/myapp
  tag: v1.2.3
replicas: 3
resources:
  requests:
    cpu: 100m
    memory: 256Mi
  limits:
    cpu: 500m
    memory: 512Mi
```

Resource requests and limits are mandatory — the admission controller rejects pods without them.

### Step 3 — Deploy with Helm

```bash
helm upgrade --install myapp ./charts/myapp -f values.yaml --namespace production
```

Verify the rollout:

```bash
kubectl rollout status deployment/myapp -n production
```

## Rollback Procedure

If a deployment fails health checks or introduces errors:

```bash
helm rollback myapp 1 --namespace production
```

This rolls back to the previous release. Always check `helm history myapp` to identify the correct revision number. Rollbacks are fast (under 30 seconds) because Helm re-applies the previous manifest.

## Canary Deployments

For high-risk changes, use a canary deployment:

1. Deploy the new version to a canary namespace with 1 replica
2. Route 5% of traffic via the Istio VirtualService weight
3. Monitor error rates and latency for 30 minutes
4. If metrics look good, promote to full production rollout
5. If issues appear, delete the canary: `kubectl delete deployment myapp-canary -n canary`

Canary traffic splitting requires Istio — see the Service Mesh section in the Infrastructure doc.

## Environment Variables and Secrets

Never hardcode secrets in Dockerfiles or Helm values. Use Kubernetes secrets:

```bash
kubectl create secret generic myapp-secrets --from-literal=DB_PASSWORD=xxx -n production
```

Reference secrets in your deployment spec via `envFrom` or `env.valueFrom.secretKeyRef`. Rotate secrets quarterly — see the Security Runbook for procedure.""",

    "Incident Response Runbook": """# Incident Response Runbook

This runbook documents the incident response process from detection through post-mortem.

## Severity Levels

| Level | Definition | Response Time | Example |
|-------|-----------|--------------|--------|
| SEV-1 | Complete service outage or data loss | 15 min | Payment system down |
| SEV-2 | Major feature degraded for >10% users | 30 min | Search returning errors |
| SEV-3 | Minor feature issue, workaround exists | 4 hours | Export button broken |
| SEV-4 | Cosmetic or low-impact issue | Next sprint | Typo in error message |

## Detection and Alerting

### PagerDuty Integration

All SEV-1 and SEV-2 alerts route through PagerDuty. The on-call engineer receives a phone call for SEV-1 and a push notification for SEV-2. Acknowledge the alert within the response time or it escalates to the engineering manager.

### Monitoring Dashboards

Key dashboards in Grafana:
- **Service Health**: `grafana.internal.io/d/service-health` — shows request rate, error rate, p50/p95/p99 latency
- **Infrastructure**: `grafana.internal.io/d/infra` — CPU, memory, disk, network for all nodes
- **Business Metrics**: `grafana.internal.io/d/business` — orders/min, signups, revenue tracking

## Incident Commander Role

The first responder becomes the Incident Commander (IC) unless they explicitly hand off. The IC's job:

1. Open the incident Slack channel: `#inc-YYYYMMDD-short-description`
2. Post the initial assessment: what's broken, who's affected, severity
3. Assign roles: IC (coordinator), Tech Lead (diagnosis), Comms Lead (stakeholders)
4. Set up a bridge call if needed (Zoom link in the channel topic)
5. Decide on mitigation strategy: rollback, feature flag, hotfix

## Mitigation Strategies

### Rollback

If the issue was introduced by a recent deploy, roll back immediately. See the Deployment Guide for Helm rollback commands. Do not debug in production — roll back first, investigate later.

### Feature Flags

If the issue is in a feature behind a flag, disable the flag via the LaunchDarkly dashboard. This is the fastest mitigation — sub-10-second propagation. All new features must be behind feature flags for exactly this reason.

### Hotfix

If rollback is not possible (e.g., database migration already applied), cut a hotfix branch from the last stable tag. Follow the expedited review process: one approval from a senior engineer, and deploy directly.

## Post-Mortem Process

Every SEV-1 and SEV-2 incident requires a blameless post-mortem within 5 business days.

### Post-Mortem Template

Use the template at `docs/templates/post-mortem.md`. Required sections:
- **Summary**: what happened, duration, impact
- **Timeline**: chronological events with timestamps
- **Root Cause**: technical explanation of the failure
- **Contributing Factors**: process or systemic issues
- **Action Items**: specific, assigned, with deadlines
- **Lessons Learned**: what should change going forward

### Blameless Culture

Post-mortems focus on systems, not individuals. We ask 'what failed?' not 'who failed?'. If the process allowed an engineer to make a mistake, the process is the root cause. Action items should prevent recurrence through automation, guardrails, or improved monitoring.""",

    "API Design Standards": """# API Design Standards

This document defines the API design standards for all REST services at the company.

## URL Structure

### Resource Naming

Use lowercase plural nouns for resources: `/users`, `/orders`, `/products`. Never use verbs in URLs — the HTTP method indicates the action.

Nested resources for parent-child relationships: `/users/{id}/orders`. Limit nesting to two levels. Deeper nesting suggests the resource should be top-level with a filter parameter.

### Versioning

Version APIs in the URL path: `/api/v1/users`. We use major versions only. Breaking changes require a new version. Non-breaking additions (new optional fields) are released within the current version.

Deprecation policy: announce 6 months before removing an old version. Include a `Sunset` header in responses from deprecated versions.

## Request and Response Format

### Content Type

All APIs accept and return `application/json`. Use `Content-Type` and `Accept` headers. Do not support XML — we standardized on JSON in 2021.

### Pagination

List endpoints must support cursor-based pagination:

```json
GET /api/v1/users?limit=20&cursor=eyJpZCI6MTAwfQ

Response:
{
  "data": [...],
  "pagination": {
    "next_cursor": "eyJpZCI6MTIwfQ",
    "has_more": true
  }
}
```

Offset-based pagination (`?page=3&per_page=20`) is acceptable for admin-facing endpoints but not for public APIs due to performance issues with large offsets.

### Error Responses

Use standard HTTP status codes and return a consistent error body:

```json
{
  "error": {
    "code": "VALIDATION_ERROR",
    "message": "Email address is not valid",
    "details": [{"field": "email", "issue": "invalid_format"}]
  }
}
```

Never expose stack traces or internal error details in production responses.

## Authentication

### Bearer Tokens

All API endpoints require authentication via Bearer token in the `Authorization` header. Tokens are JWTs issued by our auth service with a 1-hour expiry. Use the refresh token flow for long-lived sessions.

### Rate Limiting

Apply rate limiting at the API gateway level. Default limits:
- Authenticated users: 1000 requests/minute
- Unauthenticated: 60 requests/minute

Return `429 Too Many Requests` with a `Retry-After` header. Include `X-RateLimit-Remaining` and `X-RateLimit-Reset` headers in all responses.

## Documentation

Every API must have an OpenAPI 3.0 spec. Generate documentation from the spec using Redoc or Swagger UI. Keep the spec in sync with the implementation — our CI pipeline validates that the spec matches the actual routes and response schemas.""",

    "Database Operations": """# Database Operations

This document covers database management procedures for PostgreSQL and Redis.

## PostgreSQL

### Connection Management

All services connect through PgBouncer (connection pooler) at `pgbouncer.internal.io:6432`. Never connect directly to the database host — direct connections bypass connection limits and monitoring.

Connection pool settings per service:
- Default pool size: 10 connections
- Max overflow: 5 additional connections
- Connection timeout: 30 seconds
- Idle connection recycle: 300 seconds

### Migrations

We use Alembic for schema migrations. Every migration must:
1. Have both `upgrade()` and `downgrade()` functions
2. Be backwards-compatible with the previous application version (allows rollback)
3. Avoid locking large tables — use `CREATE INDEX CONCURRENTLY` instead of `CREATE INDEX`
4. Be reviewed by a database engineer for queries touching tables with >1M rows

Run migrations before deploying the new application version. This ensures the database schema is ready when the new code starts.

### Backup and Recovery

Automated daily backups via `pg_dump` run at 02:00 UTC. Point-in-time recovery (PITR) is available through WAL archiving for the last 7 days.

To restore from a backup:

```bash
pg_restore -h db-replica.internal.io -d mydb_restore backup_20240115.dump
```

Always restore to a separate database first. Verify data integrity before swapping.

### Query Performance

Monitor slow queries via the `pg_stat_statements` extension. Any query exceeding 500ms appears in the slow query dashboard at `grafana.internal.io/d/pg-slow-queries`.

Common fixes:
- Missing index: add a targeted index on the WHERE clause columns
- Sequential scan on large table: check if the query planner has up-to-date statistics (`ANALYZE tablename`)
- N+1 queries: use `JOIN` or batch loading in the application layer
- Lock contention: check `pg_locks` and consider `NOWAIT` or advisory locks

## Redis

### Usage Patterns

Redis is used for three purposes:
1. **Session storage**: user sessions with 24-hour TTL
2. **Caching**: database query results with key pattern `cache:{service}:{resource}:{id}`
3. **Rate limiting**: sliding window counters per API key

### Key Naming Convention

Use colon-separated hierarchical keys: `{purpose}:{service}:{entity}:{identifier}`. Examples:
- `session:web:user:12345`
- `cache:orderservice:order:98765`
- `ratelimit:api:key:abc123:minute`

Always set a TTL. Keys without TTL accumulate and eventually cause memory pressure.

### Monitoring

Key Redis metrics to watch:
- `used_memory` vs `maxmemory`: should stay below 80%
- `evicted_keys`: non-zero means memory pressure is causing data loss
- `connected_clients`: sudden spikes indicate connection leaks
- `keyspace_hits` / `keyspace_misses`: cache hit ratio should be >90%""",

    "Onboarding Checklist": """# Engineering Onboarding Checklist

Welcome! This checklist covers everything a new engineer needs to get productive.

## Week 1 — Environment Setup

### Access Requests

Submit access requests on Day 1 through the IT portal (`it.internal.io/access`):
- GitHub organization membership (ask your manager for team name)
- AWS SSO login (read-only initially; write access after training)
- Slack channels: #engineering, #deploys, #incidents, your team channel
- PagerDuty account (you won't be on-call for the first 4 weeks)
- Jira board access for your team

### Local Development

Clone the monorepo and follow the setup script:

```bash
git clone git@github.com:company/monorepo.git
cd monorepo && ./scripts/setup.sh
```

The setup script installs all dependencies, configures pre-commit hooks, and starts local Docker services (Postgres, Redis, Kafka). It takes about 15 minutes on a fresh machine.

Run the test suite to verify everything works:

```bash
make test
```

If tests fail, check the Troubleshooting section below or ask in #engineering.

## Week 2 — Codebase Orientation

### Architecture Overview

Our system has 12 microservices communicating via REST APIs and Kafka events. The main services:
- **user-service**: authentication, profiles, permissions
- **order-service**: order creation, payment processing, fulfillment
- **product-service**: catalog, inventory, pricing
- **notification-service**: email, SMS, push notifications
- **analytics-service**: event tracking, dashboards, reporting

Read the Architecture Decision Records (ADRs) in `docs/adr/` — they explain *why* things are built the way they are.

### Code Review Practices

Every PR needs one approval from a team member. For changes touching shared libraries or database schemas, get a second review from the platform team.

Review expectations:
- Respond to review requests within 4 hours during work hours
- Use 'Request Changes' for blocking issues, 'Comment' for suggestions
- Check for: correctness, test coverage, backwards compatibility, security

## Week 3-4 — First Contributions

### Starter Tasks

Your manager will assign 2-3 starter tasks labeled `good-first-issue` in Jira. These are scoped to take 1-2 days each and help you learn the codebase. Ask your onboarding buddy for context before starting.

### On-Call Shadow

Shadow the on-call engineer for one rotation (1 week) before going on-call yourself. During shadowing:
- Join the incident bridge calls
- Review the monitoring dashboards daily
- Read recent post-mortems (last 3 months) in `docs/post-mortems/`

## Troubleshooting

### Common Setup Issues

| Problem | Solution |
|---------|----------|
| Docker daemon not running | Start Docker Desktop; on Linux: `sudo systemctl start docker` |
| Port 5432 already in use | Kill local postgres: `sudo lsof -ti:5432 \| xargs kill` |
| npm install fails | Delete `node_modules` and `package-lock.json`, then retry |
| Pre-commit hooks slow | Run `pre-commit gc` to clean cache |
| AWS credentials expired | Re-authenticate: `aws sso login --profile default` |

If none of these help, post in #engineering with the full error output.""",
}

print(f"Documents: {len(WIKI_DOCS)}")
for title, content in WIKI_DOCS.items():
    lines = content.strip().split("\n")
    headings = [l for l in lines if l.startswith("#")]
    words = len(content.split())
    print(f"  {title:30s} | {words:>5} words | {len(headings)} headings")

---

# Part B — Markdown-Aware Parsing

## 11. Heading-Based Splitter

Instead of splitting by character count, we split on heading boundaries. Each chunk becomes a section with the full heading path preserved.

In [ ]:
def parse_markdown_sections(doc_title: str, markdown_text: str) -> list[dict]:
    """Split markdown into sections based on heading hierarchy.

    Returns a list of dicts, each with:
    - text: the section body (no heading line)
    - heading: the heading text
    - level: heading level (1, 2, 3)
    - section_path: full path like 'Doc Title > H1 > H2 > H3'
    - doc_title: source document name
    - word_count: words in the section body
    """
    lines = markdown_text.strip().split("\n")
    sections = []
    heading_stack = []  # [(level, heading_text), ...]
    current_heading = doc_title
    current_level = 0
    current_lines = []

    def flush_section():
        body = "\n".join(current_lines).strip()
        if body:  # skip empty sections
            path_parts = [doc_title] + [h[1] for h in heading_stack]
            sections.append({
                "text": body,
                "heading": current_heading,
                "level": current_level,
                "section_path": " > ".join(path_parts),
                "doc_title": doc_title,
                "word_count": len(body.split()),
            })

    for line in lines:
        heading_match = re.match(r"^(#{1,6})\s+(.+)$", line)
        if heading_match:
            # Flush previous section
            flush_section()
            current_lines = []

            level = len(heading_match.group(1))
            heading_text = heading_match.group(2).strip()
            current_heading = heading_text
            current_level = level

            # Update heading stack: pop deeper or equal headings
            while heading_stack and heading_stack[-1][0] >= level:
                heading_stack.pop()
            heading_stack.append((level, heading_text))
        else:
            current_lines.append(line)

    # Flush last section
    flush_section()
    return sections


# Test on one document
test_sections = parse_markdown_sections("Deployment Guide", WIKI_DOCS["Deployment Guide"])
print(f"Sections parsed from 'Deployment Guide': {len(test_sections)}")
for s in test_sections:
    print(f"  L{s['level']} ({s['word_count']:>3} words) | {s['section_path']}")

## 12. Parse All Documents

In [ ]:
all_sections = []
for doc_title, markdown_text in WIKI_DOCS.items():
    sections = parse_markdown_sections(doc_title, markdown_text)
    all_sections.extend(sections)

print(f"Total sections across all docs: {len(all_sections)}")
print(f"\nSections per document:")
for doc in WIKI_DOCS:
    count = sum(1 for s in all_sections if s["doc_title"] == doc)
    words = sum(s["word_count"] for s in all_sections if s["doc_title"] == doc)
    print(f"  {doc:30s} | {count:>2} sections | {words:>5} words")

print(f"\nHeading level distribution:")
for level, count in sorted(Counter(s["level"] for s in all_sections).items()):
    print(f"  H{level}: {count} sections")

print(f"\nWord count stats:")
wc = [s["word_count"] for s in all_sections]
print(f"  Min: {min(wc)}, Max: {max(wc)}, Avg: {sum(wc)//len(wc)}, Total: {sum(wc)}")

## 13. Inspect Parsed Sections

In [ ]:
print("ALL PARSED SECTIONS")
print("=" * 110)
for i, s in enumerate(all_sections):
    preview = s["text"][:80].replace("\n", " ")
    print(f"  [{i:>2}] L{s['level']} {s['word_count']:>3}w | {s['section_path'][:55]:55s} | {preview}...")

---

# Part C — Embedding & Indexing

## 14. Build the Vector Index

Each section is embedded and stored in ChromaDB with its full metadata — document title, heading path, level, and word count.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("wiki_sections")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="wiki_sections",
    metadata={"hnsw:space": "cosine"},
)

# Prepare data for indexing
section_texts = []
section_metadatas = []
section_ids = []

for i, s in enumerate(all_sections):
    # Embed the heading + body together for better semantic matching
    embed_text = f"{s['section_path']}\n{s['text']}"
    section_texts.append(embed_text)
    section_metadatas.append({
        "doc_title": s["doc_title"],
        "heading": s["heading"],
        "level": s["level"],
        "section_path": s["section_path"],
        "word_count": s["word_count"],
    })
    section_ids.append(f"sec-{i:03d}")

vectors = embeddings.embed_documents(section_texts)

collection.add(
    documents=section_texts,
    embeddings=vectors,
    metadatas=section_metadatas,
    ids=section_ids,
)

print(f"Indexed {collection.count()} sections in ChromaDB")
print(f"Embedding dim: {len(vectors[0])}")
print(f"Avg embed text length: {sum(len(t) for t in section_texts) // len(section_texts)} chars")

---

# Part D — Retrieval With Metadata Filters

## 15. Core Retrieval Function

In [ ]:
def retrieve_sections(query: str, top_k: int = TOP_K,
                      doc_title: Optional[str] = None,
                      min_level: Optional[int] = None,
                      max_level: Optional[int] = None) -> list[dict]:
    """Retrieve sections with optional metadata filters."""
    query_vector = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_vector], "n_results": top_k}

    conditions = []
    if doc_title:
        conditions.append({"doc_title": doc_title})
    if min_level:
        conditions.append({"level": {"$gte": min_level}})
    if max_level:
        conditions.append({"level": {"$lte": max_level}})

    if len(conditions) == 1:
        kwargs["where"] = conditions[0]
    elif len(conditions) > 1:
        kwargs["where"] = {"$and": conditions}

    results = collection.query(**kwargs)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
            "similarity": 1 - results["distances"][0][i],
        })
    return hits


def display_hits(hits: list[dict], label: str = ""):
    if label:
        print(f"\n  [{label}]")
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        print(f"    {i}. sim={h['similarity']:.3f} | L{m['level']} {m['word_count']:>3}w "
              f"| {m['section_path'][:60]}")


# Test retrieval
print("Test: unfiltered retrieval")
q = "how to roll back a failed deployment"
print(f"Q: {q}")
hits = retrieve_sections(q, top_k=3)
display_hits(hits, "All docs")

## 16. Demonstrate Metadata-Filtered Retrieval

In [ ]:
q = "how to handle a production outage"
print(f"Q: {q}")

print("\n--- No filters ---")
display_hits(retrieve_sections(q, top_k=3), "All")

print("\n--- Filter: Incident Response Runbook only ---")
display_hits(retrieve_sections(q, top_k=3, doc_title="Incident Response Runbook"), "Incident only")

print("\n--- Filter: Top-level sections (H1/H2) only ---")
display_hits(retrieve_sections(q, top_k=3, max_level=2), "H1/H2 only")

In [ ]:
q = "database connection pooling configuration"
print(f"Q: {q}")

print("\n--- No filters ---")
display_hits(retrieve_sections(q, top_k=3), "All")

print("\n--- Filter: Database Operations only ---")
display_hits(retrieve_sections(q, top_k=3, doc_title="Database Operations"), "DB ops")

In [ ]:
q = "how does API authentication work"
print(f"Q: {q}")

print("\n--- No filters ---")
display_hits(retrieve_sections(q, top_k=3), "All")

print("\n--- Filter: API Design Standards only ---")
display_hits(retrieve_sections(q, top_k=3, doc_title="API Design Standards"), "API standards")

In [ ]:
q = "new engineer first week tasks"
print(f"Q: {q}")

print("\n--- No filters ---")
display_hits(retrieve_sections(q, top_k=3), "All")

print("\n--- Filter: Onboarding Checklist + detail sections (H3) ---")
display_hits(
    retrieve_sections(q, top_k=3, doc_title="Onboarding Checklist", min_level=3),
    "Onboarding H3"
)

---

# Part E — QA With Section Citations

## 17. Retrieval-Augmented QA Pipeline

In [ ]:
QA_SYSTEM = """You are a documentation assistant that answers questions using internal wiki articles.

Rules:
1. Answer ONLY from the provided context sections. Do not add information not in the context.
2. Cite every fact as [Doc: section path]. Example: [Doc: Deployment Guide > Rollback Procedure]
3. If the context does not contain the answer, say 'Not found in the indexed documentation.'
4. Be concise and direct — engineers want answers, not prose.
5. If the question spans multiple documents, synthesize across them and cite each."""


def build_context(hits: list[dict]) -> str:
    parts = []
    for h in hits:
        m = h["metadata"]
        # Use the raw section text (without the path prefix used for embedding)
        section_idx = next(
            (i for i, s in enumerate(all_sections)
             if s["section_path"] == m["section_path"] and s["doc_title"] == m["doc_title"]),
            None
        )
        body = all_sections[section_idx]["text"] if section_idx is not None else h["text"]
        parts.append(f"[Section: {m['section_path']}]\n{body}")
    return "\n\n---\n\n".join(parts)


def wiki_qa(question: str, top_k: int = TOP_K,
            doc_title: Optional[str] = None,
            min_level: Optional[int] = None,
            max_level: Optional[int] = None) -> dict:
    """Full RAG pipeline: retrieve sections -> generate answer with citations."""
    hits = retrieve_sections(question, top_k=top_k,
                             doc_title=doc_title,
                             min_level=min_level,
                             max_level=max_level)
    context = build_context(hits)

    prompt = (
        f"Answer this question using the wiki sections below.\n\n"
        f"QUESTION: {question}\n\n"
        f"CONTEXT SECTIONS:\n{context}\n\n"
        "Return ONLY JSON:\n"
        '{"answer": "your answer with [Doc: section path] citations",'
        ' "sources": ["section path 1", "section path 2"],'
        ' "confidence": "high|medium|low"}'
    )

    resp = ask(prompt, system=QA_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {
            "answer": resp,
            "sources": [h["metadata"]["section_path"] for h in hits[:3]],
            "confidence": "unknown",
        }
    result["hits"] = hits
    return result


def display_answer(question: str, result: dict):
    print(f"\n{'='*90}")
    print(f"Q: {question}")
    conf = result.get("confidence", "?")
    print(f"Confidence: {conf}")
    print(f"{'='*90}")
    print(f"\nA:")
    wrap_print(result.get("answer", ""))
    if result.get("sources"):
        print(f"\nSources:")
        for src in result["sources"]:
            print(f"  - {src}")


print("QA pipeline ready")

## 18. QA — Deployment Questions

In [ ]:
q = "How do I roll back a failed Helm deployment?"
r = wiki_qa(q)
display_answer(q, r)

In [ ]:
q = "What is the canary deployment process?"
r = wiki_qa(q, doc_title="Deployment Guide")
display_answer(q, r)

## 19. QA — Incident Response Questions

In [ ]:
q = "What are the severity levels and their response times?"
r = wiki_qa(q, doc_title="Incident Response Runbook")
display_answer(q, r)

In [ ]:
q = "What should I do as incident commander?"
r = wiki_qa(q)
display_answer(q, r)

## 20. QA — Cross-Document Questions

In [ ]:
q = "If a deployment fails, what are all the ways I can fix it — from the deployment guide and the incident runbook?"
r = wiki_qa(q, top_k=6)
display_answer(q, r)

In [ ]:
q = "What monitoring dashboards are available and what do they show?"
r = wiki_qa(q, top_k=5)
display_answer(q, r)

## 21. QA — Database & API Questions

In [ ]:
q = "How should I name Redis keys and what TTL should I set?"
r = wiki_qa(q, doc_title="Database Operations")
display_answer(q, r)

In [ ]:
q = "What pagination format should I use in APIs?"
r = wiki_qa(q, doc_title="API Design Standards")
display_answer(q, r)

## 22. QA — Onboarding Questions

In [ ]:
q = "I just joined the team — what access do I need to request on day 1?"
r = wiki_qa(q, doc_title="Onboarding Checklist")
display_answer(q, r)

In [ ]:
q = "Docker daemon won't start and port 5432 is busy — how do I fix the local setup?"
r = wiki_qa(q)
display_answer(q, r)

---

# Part F — Retrieval Evaluation

## 23. Ground-Truth Evaluation Set

In [ ]:
EVAL_SET = [
    {"query": "how to build and push a Docker image",
     "expected_path": "Deployment Guide > Standard Deployment Process > Step 1 — Build and Push Image",
     "filter_doc": "Deployment Guide"},
    {"query": "Helm rollback command",
     "expected_path": "Deployment Guide > Rollback Procedure",
     "filter_doc": "Deployment Guide"},
    {"query": "canary deployment with Istio",
     "expected_path": "Deployment Guide > Canary Deployments",
     "filter_doc": "Deployment Guide"},
    {"query": "kubernetes secrets for environment variables",
     "expected_path": "Deployment Guide > Environment Variables and Secrets",
     "filter_doc": "Deployment Guide"},
    {"query": "SEV-1 vs SEV-2 response time",
     "expected_path": "Incident Response Runbook > Severity Levels",
     "filter_doc": "Incident Response Runbook"},
    {"query": "PagerDuty alert escalation",
     "expected_path": "Incident Response Runbook > Detection and Alerting > PagerDuty Integration",
     "filter_doc": "Incident Response Runbook"},
    {"query": "incident commander responsibilities",
     "expected_path": "Incident Response Runbook > Incident Commander Role",
     "filter_doc": "Incident Response Runbook"},
    {"query": "post-mortem blameless culture",
     "expected_path": "Incident Response Runbook > Post-Mortem Process > Blameless Culture",
     "filter_doc": "Incident Response Runbook"},
    {"query": "API URL naming convention plural nouns",
     "expected_path": "API Design Standards > URL Structure > Resource Naming",
     "filter_doc": "API Design Standards"},
    {"query": "cursor-based pagination format",
     "expected_path": "API Design Standards > Request and Response Format > Pagination",
     "filter_doc": "API Design Standards"},
    {"query": "rate limiting headers and limits",
     "expected_path": "API Design Standards > Authentication > Rate Limiting",
     "filter_doc": "API Design Standards"},
    {"query": "PgBouncer connection pool settings",
     "expected_path": "Database Operations > PostgreSQL > Connection Management",
     "filter_doc": "Database Operations"},
    {"query": "Alembic database migration rules",
     "expected_path": "Database Operations > PostgreSQL > Migrations",
     "filter_doc": "Database Operations"},
    {"query": "Redis key naming convention",
     "expected_path": "Database Operations > Redis > Key Naming Convention",
     "filter_doc": "Database Operations"},
    {"query": "new hire access requests day 1",
     "expected_path": "Onboarding Checklist > Week 1 — Environment Setup > Access Requests",
     "filter_doc": "Onboarding Checklist"},
    {"query": "monorepo setup script and test command",
     "expected_path": "Onboarding Checklist > Week 1 — Environment Setup > Local Development",
     "filter_doc": "Onboarding Checklist"},
    {"query": "code review approval requirements",
     "expected_path": "Onboarding Checklist > Week 2 — Codebase Orientation > Code Review Practices",
     "filter_doc": "Onboarding Checklist"},
    {"query": "Docker port 5432 conflict fix",
     "expected_path": "Onboarding Checklist > Troubleshooting > Common Setup Issues",
     "filter_doc": "Onboarding Checklist"},
]

print(f"Evaluation set: {len(EVAL_SET)} query-section pairs")
print(f"Documents covered: {len(set(e['filter_doc'] for e in EVAL_SET))}")

## 24. Run Evaluation: Filtered vs Unfiltered

In [ ]:
def run_eval(eval_set: list[dict], use_filter: bool, top_k: int = 5) -> dict:
    hits_at_k = 0
    recip_ranks = []
    details = []

    for item in eval_set:
        kwargs = {}
        if use_filter:
            kwargs["doc_title"] = item["filter_doc"]

        hits = retrieve_sections(item["query"], top_k=top_k, **kwargs)
        paths = [h["metadata"]["section_path"] for h in hits]
        expected = item["expected_path"]

        found_rank = None
        for rank, path in enumerate(paths, 1):
            if path == expected:
                found_rank = rank
                break

        if found_rank is not None:
            hits_at_k += 1
            recip_ranks.append(1.0 / found_rank)
        else:
            recip_ranks.append(0.0)

        details.append({
            "query": item["query"],
            "expected": expected,
            "top_3": paths[:3],
            "hit": found_rank is not None,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "recall_at_k": hits_at_k / n,
        "mrr": sum(recip_ranks) / n,
        "n": n,
        "k": top_k,
        "details": details,
    }


eval_unf = run_eval(EVAL_SET, use_filter=False, top_k=5)
eval_flt = run_eval(EVAL_SET, use_filter=True, top_k=5)

print("RETRIEVAL EVALUATION")
print("=" * 55)
print(f"{'Metric':<20} {'Unfiltered':>12} {'Filtered':>12}")
print("-" * 55)
print(f"{'Recall@5':<20} {eval_unf['recall_at_k']:>11.1%} {eval_flt['recall_at_k']:>11.1%}")
print(f"{'MRR':<20} {eval_unf['mrr']:>11.3f} {eval_flt['mrr']:>11.3f}")
print(f"{'Questions':<20} {eval_unf['n']:>12} {eval_flt['n']:>12}")

In [ ]:
print("\nPER-QUERY COMPARISON")
print("=" * 100)

fw = uw = t = 0
for ud, fd in zip(eval_unf["details"], eval_flt["details"]):
    u_r = ud["rank"] or 999
    f_r = fd["rank"] or 999
    if f_r < u_r:
        tag = "FILT"; fw += 1
    elif u_r < f_r:
        tag = "UNFL"; uw += 1
    else:
        tag = "TIE "; t += 1
    u_s = f"@{ud['rank']}" if ud["hit"] else "MISS"
    f_s = f"@{fd['rank']}" if fd["hit"] else "MISS"
    print(f"  [{tag}] U={u_s:>5} F={f_s:>5} | {ud['query'][:60]}")

print(f"\nFiltered wins: {fw} | Unfiltered wins: {uw} | Ties: {t}")

## 25. Recall at Different k

In [ ]:
print("RECALL AT DIFFERENT K")
print("=" * 60)
print(f"{'k':>3} | {'Unfilt R@k':>12} {'Unfilt MRR':>12} | {'Filt R@k':>12} {'Filt MRR':>12}")
print("-" * 60)
for k in [1, 3, 5]:
    u = run_eval(EVAL_SET, use_filter=False, top_k=k)
    f = run_eval(EVAL_SET, use_filter=True, top_k=k)
    print(f"  {k:>1} | {u['recall_at_k']:>11.1%} {u['mrr']:>11.3f} | {f['recall_at_k']:>11.1%} {f['mrr']:>11.3f}")

## 26. Answer Quality — LLM-as-Judge

In [ ]:
JUDGE_SYSTEM = "You evaluate documentation QA answers for quality."

JUDGE_PROMPT = """Evaluate this answer to a wiki/documentation question.

QUESTION: {question}
ANSWER: {answer}
CITED SOURCES: {sources}

Score each dimension 1-5:
- correctness: does the answer accurately reflect the source documentation?
- completeness: does it cover the key points from the relevant section?
- citations: are sources properly cited with section paths?
- conciseness: is it appropriately brief for an engineer looking up docs?

Return ONLY JSON:
{{"correctness": N, "completeness": N, "citations": N, "conciseness": N,
  "overall": N, "notes": "brief explanation"}}"""

judge_questions = [
    "How do I roll back a failed Helm deployment?",
    "What are the severity levels and their response times?",
    "How should I name Redis keys?",
    "What access do I need as a new engineer on day 1?",
]

print("ANSWER QUALITY EVALUATION (LLM-as-Judge)")
print("=" * 80)

for q in judge_questions:
    result = wiki_qa(q)
    sources_str = ", ".join(result.get("sources", [])[:3])
    resp = ask(
        JUDGE_PROMPT.format(
            question=q,
            answer=str(result.get("answer", ""))[:400],
            sources=sources_str,
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Q: {q[:60]}")
        for dim in ["correctness", "completeness", "citations", "conciseness"]:
            val = scores.get(dim, "?")
            bar = "*" * int(val) if isinstance(val, (int, float)) else "?"
            print(f"    {dim:16s}: {val}/5 {bar}")
        print(f"    {'overall':16s}: {scores.get('overall', '?')}/5")
        if scores.get("notes"):
            print(f"    Notes: {scores['notes'][:80]}")

---

## 27. Edge Cases

In [ ]:
print("EDGE CASE 1: Question about a topic not in the wiki")
q = "How do I configure the Kafka consumer group offsets?"
r = wiki_qa(q)
display_answer(q, r)

print("\n" + "="*90)
print("EDGE CASE 2: Very broad question")
q = "Tell me everything about our infrastructure"
r = wiki_qa(q, top_k=6)
display_answer(q, r)

print("\n" + "="*90)
print("EDGE CASE 3: Wrong document filter")
q = "How do I deploy with Helm?"
r = wiki_qa(q, doc_title="Database Operations")
display_answer(q, r)

## 28. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Splitting markdown by character count | Breaks mid-sentence; loses heading context | Split on heading boundaries |
| Not preserving heading hierarchy | Chunks lose their location in the doc | Track `section_path` as `Doc > H1 > H2 > H3` |
| Embedding only the section body | The heading itself carries key semantic info | Embed `section_path + body` together |
| Ignoring empty sections | Empty heading-only sections waste index space | Filter out sections with no body text |
| No metadata in ChromaDB | Can't filter by document or heading level | Store `doc_title`, `level`, `section_path` as metadata |
| Returning full documents | Too much context; answer gets diluted | Return individual sections; smaller context = better answers |

## 29. Production Improvement Ideas

1. **File system watcher** — auto-re-index when markdown files change on disk
2. **Git-aware versioning** — track which commit each section was indexed from; detect stale docs
3. **Cross-references** — detect "see the Deployment Guide" links and follow them during retrieval
4. **Table parsing** — extract and index markdown tables as structured data
5. **Code block extraction** — index code examples separately with language tags for code search
6. **Incremental re-indexing** — only re-embed sections whose content hash changed
7. **User feedback loop** — track which sections users click; boost frequently-helpful sections

## 30. Exercises

### Exercise 1
Add a 6th wiki document (e.g., "Testing Standards" or "CI/CD Pipeline"). Write 3 evaluation query-section pairs and verify that retrieval remains accurate for both old and new documents.

### Exercise 2
Implement a `search_within_section` function: given a broad section (e.g., "Database Operations > PostgreSQL"), further split long sections into paragraphs and do a second-pass retrieval within the section. Compare precision with single-pass retrieval.

### Exercise 3
Build a citation verifier: after generating an answer, extract all `[Doc: ...]` citations, look up each one in the index, and report whether the cited content actually supports the claim. Flag unsupported citations.

### Mini Challenge
Add **automatic document classification**: when a new markdown doc is added, use the LLM to classify it into a category (runbook, guide, standard, reference, tutorial). Store the category as metadata and allow filtering by document type during retrieval.

In [ ]:
print("SESSION SUMMARY")
print("=" * 60)
print(f"Documents indexed: {len(WIKI_DOCS)}")
print(f"Total sections: {len(all_sections)}")
print(f"Total words: {sum(s['word_count'] for s in all_sections):,}")
print(f"Evaluation pairs: {len(EVAL_SET)}")
print(f"\nRetrieval (Recall@5 / MRR):")
print(f"  Unfiltered: {eval_unf['recall_at_k']:.1%} / {eval_unf['mrr']:.3f}")
print(f"  Filtered:   {eval_flt['recall_at_k']:.1%} / {eval_flt['mrr']:.3f}")
print(f"\nCapabilities built:")
print(f"  - Markdown heading-aware parser with section path tracking")
print(f"  - Rich metadata indexing (doc, heading, level, word count)")
print(f"  - Metadata-filtered retrieval (by document, heading level)")
print(f"  - Section-cited QA answers with [Doc: path] references")
print(f"  - Retrieval evaluation (Recall@k, MRR, per-query comparison)")
print(f"  - LLM-as-judge quality assessment")

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Heading-based splitting** preserves document structure better than character-count chunking |
| 2 | **Section paths** (`Doc > H1 > H2`) enable precise citations and improve semantic matching |
| 3 | **Embed heading + body together** — the heading carries crucial context about the section's topic |
| 4 | **Metadata filters** (by document, heading level) improve precision when users know where to look |
| 5 | **Section-level retrieval** beats page-level: smaller chunks = more focused answers |
| 6 | Empty sections (heading only, no body) should be **filtered out** during indexing |
| 7 | Cross-document questions work when **top-k is high enough** to pull from multiple docs |
| 8 | Retrieval evaluation with **ground-truth pairs** catches regressions when you change chunking |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*